# 03 — Fehlalarme und zeitliche Plausibilisierung

Entspricht Kapitel 5, Abschnitte *Falsch-Positiv-Test*,
*N-von-M-Voting* und *Hard-Negative-Training*.

Fuer den praktischen Einsatz ist die Fehlalarmrate wichtiger als die
Sensitivitaet: ein System, das bei jedem zweiten Schritt Alarm
ausloest, ist unbrauchbar.

> **Zwei Korrekturen gegenueber frueheren Laeufen:**
> 1. Fuer das Voting betraegt die Schrittweite 0,2 s. Bei 1,0 s liegt
>    der Aufprall in nur einem Fenster, sodass jede Regel mit N ≥ 2
>    zwangslaeufig scheitert.
> 2. Innerhalb einer Sturzaufzeichnung gilt nur das Fenster als
>    Sturz, das den Aufprall enthaelt. Zuvor wurden alle Fenster
>    pauschal als Sturz beschriftet.

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q scipy scikit-learn tensorflow openpyxl

import os, sys, shutil, time, json
from pathlib import Path
import numpy as np
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# Gemeinsame Bibliothek holen (einzige Quelle fuer alle Loader)
!wget -q https://raw.githubusercontent.com/Alyelb/ML-Modell-for-Fall-detection/main/fall_lib.py -O /content/fall_lib.py
sys.path.insert(0, "/content")
import fall_lib as fl

BASE  = "/content/drive/MyDrive/ML_Fall_Detection"
fl.RESULTS_ROOT = f"{BASE}/results"
MODEL = f"{BASE}/results/fall_model.keras"      # einheitlicher Modellpfad

print("Konfiguration:", fl.default_config())

Mounted at /content/drive
Konfiguration: {'fs_target': 50, 'window_sec': 2.0, 'overlap': 0.5, 'window_len': 100, 'step': 50, 'kfall_fall_tasks': 'T20-T34'}


In [2]:
def hole(name, ziel):
    """Kopiert einen Datensatz von Drive auf die lokale Platte.
    Drive ist fuer tausende Kleindateien zu langsam."""
    src = f"{BASE}/{name}"
    if Path(ziel).exists():
        print(f"[ok] {name} bereits lokal"); return ziel
    if not Path(src).exists():
        print(f"[!] {src} nicht gefunden. Vorhanden:")
        for p in sorted(Path(BASE).iterdir()):
            print("   ", p.name)
        return None
    t = time.time(); shutil.copytree(src, ziel)
    print(f"[ok] {name} kopiert in {time.time()-t:.0f}s")
    return ziel

In [3]:
from tensorflow import keras
model = keras.models.load_model(MODEL, compile=False)
model.compile(optimizer="adam", loss="binary_crossentropy",
              metrics=["accuracy"])
run = fl.new_run("03_fehlalarme")
ergebnisse = {}

POSITION = "ANKLE"     # fussnaechste verfuegbare Position
UMA = hole("UMAFall Fall Detection/UMAFall_Dataset_corrected",
           "/content/UMAFall")
trials = fl.load_umafall(UMA, position=POSITION)

[Lauf] /content/drive/MyDrive/ML_Fall_Detection/results/03_fehlalarme_2026-08-26_1102
[ok] UMAFall Fall Detection/UMAFall_Dataset_corrected kopiert in 32s
[UMAFall ANKLE] 643 Trials (157 Sturz, 486 ADL), 19 Probanden


## Falsch-Positiv-Test auf fussnahen Alltagsdaten

Ausgewertet werden ausschliesslich ADL-Aufzeichnungen. Da keine
Stuerze enthalten sind, ist jede positive Klassifikation per
Definition ein Fehlalarm.

In [4]:
adl = [t for t in trials if t["label"] == 0]
X, y, g, acts = fl.build_windows(adl, mode="sliding")
X = fl.znorm(X)
probs = model.predict(X, batch_size=256, verbose=0).flatten()

stunden = len(X) * fl.WIN_SEC * (1 - fl.OVERLAP) / 3600
print(f"{len(X)} ADL-Fenster = {stunden:.2f} Stunden\n")
print(f"{'Schwelle':>9} {'FP':>7} {'Rate':>8} {'FP/Std':>9}")
print("-"*36)
schwellen = {}
for thr in [0.5, 0.7, 0.9, 0.95, 0.99]:
    fp = int((probs > thr).sum())
    print(f"{thr:>9.2f} {fp:>7d} {100*fp/len(probs):>7.2f}% "
          f"{fp/stunden:>9.1f}")
    schwellen[str(thr)] = {"fp": fp, "rate": round(fp/len(probs), 5),
                           "fp_per_hour": round(fp/stunden, 2)}
ergebnisse["fp_schwellen"] = schwellen

6329 ADL-Fenster = 1.76 Stunden

 Schwelle      FP     Rate    FP/Std
------------------------------------
     0.50     165    2.61%      93.9
     0.70     131    2.07%      74.5
     0.90      86    1.36%      48.9
     0.95      69    1.09%      39.2
     0.99      39    0.62%      22.2


### Aufschluesselung nach Aktivitaet

In [5]:
print(f"{'Aktivitaet':<26} {'Fenster':>8} {'FP':>6} {'Rate':>8} {'max p':>7}")
print("-"*60)
nach_akt = {}
for a in sorted(np.unique(acts)):
    p = probs[acts == a]
    fp = int((p > 0.5).sum())
    print(f"{a:<26} {len(p):>8d} {fp:>6d} {100*fp/len(p):>7.2f}% "
          f"{p.max():>7.3f}")
    nach_akt[a] = {"windows": int(len(p)), "fp": fp,
                   "rate": round(fp/len(p), 4),
                   "max_prob": round(float(p.max()), 4)}
ergebnisse["fp_nach_aktivitaet"] = nach_akt

Aktivitaet                  Fenster     FP     Rate   max p
------------------------------------------------------------
Aplausing                       548      3    0.55%   0.925
Bending                         648      7    1.08%   0.999
GoDownstairs                    259      7    2.70%   0.997
GoUpstairs                      255      6    2.35%   0.999
HandsUp                         567     11    1.94%   1.000
Hopping                         584     10    1.71%   0.999
Jogging                         259     40   15.44%   0.999
LyingDown_OnABed                624     13    2.08%   1.000
MakingACall                     610     11    1.80%   1.000
OpeningDoor                     563     19    3.37%   1.000
Sitting_GettingUpOnAChair       699     12    1.72%   1.000
Walking                         713     26    3.65%   0.999


## N-von-M-Voting

Ein Alarm wird erst ausgeloest, wenn innerhalb der letzten M
Entscheidungen mindestens N positiv ausfallen. Die Schrittweite
betraegt hier 0,2 s (90 % Ueberlappung), damit der Aufprall in
mehreren aufeinanderfolgenden Fenstern erscheint.

In [6]:
STEP_FEIN = int(fl.FS_TARGET * 0.2)      # 10 Abtastwerte = 0,2 s
SEC_FEIN  = 0.2

def trial_probs(m, trials, step):
    """Wahrscheinlichkeitsfolgen je Trial, Reihenfolge erhalten."""
    seqs, fenster = [], []
    for t in trials:
        w = fl.window_sliding(t["data"], fl.WIN_LEN, step)
        if not w: continue
        fenster.append((fl.znorm(np.asarray(w, np.float32)),
                        bool(t["label"])))
    alle = np.concatenate([f for f, _ in fenster], axis=0)
    p = m.predict(alle, batch_size=512, verbose=0).flatten()
    off = 0
    for f, is_fall in fenster:
        seqs.append((p[off:off+len(f)], is_fall)); off += len(f)
    return seqs

print("Berechne Wahrscheinlichkeitsfolgen ...")
seqs_basis = trial_probs(model, trials, STEP_FEIN)
print(f"{len(seqs_basis)} Trials")

Berechne Wahrscheinlichkeitsfolgen ...
643 Trials


### Voting braucht ein angepasstes Modell

Verglichen wird das unveraenderte Basismodell mit einem auf
Knoecheldaten feinabgestimmten Modell. Der Unterschied zeigt, ob
Voting die Domaenenanpassung ersetzen kann.

In [7]:
# Fine-Tuning auf einem Teil der Probanden
Xa, ya, ga, _ = fl.build_windows(trials, mode="sliding")
Xa = fl.znorm(Xa)
tr_i, te_i = fl.subject_split(ga, ya, test_size=0.3)
test_subs = set(np.unique(ga[te_i]))

model_ft = fl.finetune(model, Xa[tr_i], ya[tr_i])
print("Fine-Tuning abgeschlossen")

test_trials = [t for t in trials if t["subject"] in test_subs]
seqs = {"basis": trial_probs(model, test_trials, STEP_FEIN),
        "ft":    trial_probs(model_ft, test_trials, STEP_FEIN)}
print(f"Testsatz: {len(test_trials)} Trials")

Fine-Tuning abgeschlossen
Testsatz: 206 Trials


In [8]:
regeln = [(1,1), (2,3), (3,5), (5,7), (7,10)]
voting = {}
for tag, name in [("basis","Basismodell"), ("ft","Feinabgestimmt")]:
    print(f"\n{name}")
    print(f"  {'Regel':>9} {'Schwelle':>9} {'Se':>7} {'FP/Std':>9}")
    print("  " + "-"*38)
    for n, m_ in regeln:
        for thr in [0.5, 0.9]:
            r = fl.voting_eval(seqs[tag], thr, n, m_, SEC_FEIN)
            print(f"  {r['rule']:>9} {thr:>9.2f} "
                  f"{(r['Se'] if r['Se'] is not None else 0):>7.3f} "
                  f"{(r['fp_per_hour'] or 0):>9.1f}")
            voting[f"{tag}_{n}of{m_}@{thr}"] = r
ergebnisse["voting"] = voting


Basismodell
      Regel  Schwelle      Se    FP/Std
  --------------------------------------
    1-von-1      0.50   0.511     235.2
    1-von-1      0.90   0.362     128.1
    2-von-3      0.50   0.425     114.1
    2-von-3      0.90   0.234      47.4
    3-von-5      0.50   0.319      64.9
    3-von-5      0.90   0.085      29.8
    5-von-7      0.50   0.064      24.6
    5-von-7      0.90   0.043      17.6
   7-von-10      0.50   0.021      10.5
   7-von-10      0.90   0.021      12.3

Feinabgestimmt
      Regel  Schwelle      Se    FP/Std
  --------------------------------------
    1-von-1      0.50   0.617     265.0
    1-von-1      0.90   0.468     175.5
    2-von-3      0.50   0.511     143.9
    2-von-3      0.90   0.340      79.0
    3-von-5      0.50   0.468     100.0
    3-von-5      0.90   0.319      43.9
    5-von-7      0.50   0.277      42.1
    5-von-7      0.90   0.128      24.6
   7-von-10      0.50   0.064      14.0
   7-von-10      0.90   0.043       8.8


## Hard-Negative-Training

Alltagsbewegungen mit hohen Beschleunigungsspitzen (Gehen, Joggen,
Treppen) sind die eigentliche Fehlerquelle. Getestet wird, ob eine
Hoehergewichtung dieser Fenster im Training hilft.

In [9]:
HARD_WEIGHT, PEAK_THR = 3.0, 1.8      # Faktor, Schwelle in g

Xr, yr, gr, _ = fl.build_windows(trials, mode="sliding")
peaks = fl.acc_magnitude(Xr).max(axis=1)     # vor der Normalisierung
Xr = fl.znorm(Xr)

w = np.ones(len(yr), np.float32)
hart = (yr == 0) & (peaks > PEAK_THR)
w[hart] = HARD_WEIGHT
cw = fl.class_weights(yr)
w[yr == 1] *= cw[1]; w[yr == 0] *= cw[0]
print(f"Hard Negatives: {int(hart.sum())} von {int((yr==0).sum())} "
      f"ADL-Fenstern ({100*hart.sum()/max((yr==0).sum(),1):.1f}%)")

m_ohne = fl.finetune(model, Xr[tr_i], yr[tr_i])
m_mit  = fl.finetune(model, Xr[tr_i], yr[tr_i], sample_weight=w[tr_i])

print()
hn = {}
for m_, nm in [(model,"Basismodell"), (m_ohne,"FT ohne Hard Neg."),
               (m_mit,"FT mit Hard Neg.")]:
    hn[nm] = fl.evaluate(m_, Xr[te_i], yr[te_i], nm)
ergebnisse["hard_negatives"] = {
    "config": {"weight": HARD_WEIGHT, "peak_g": PEAK_THR,
               "n_hard": int(hart.sum())}, "results": hn}

fl.save_results(run, ergebnisse, config=fl.default_config(),
    notes=("Voting mit 0,2 s Schrittweite; Sturzfenster nur dort, wo "
           "der Aufprall liegt."))

Hard Negatives: 2160 von 8070 ADL-Fenstern (26.8%)

  Basismodell                 Se=0.141  Sp=0.976  Acc=0.948  FP=   62  FN=  79
  FT ohne Hard Neg.           Se=0.261  Sp=0.966  Acc=0.942  FP=   87  FN=  68
  FT mit Hard Neg.            Se=0.696  Sp=0.950  Acc=0.941  FP=  130  FN=  28
[gesichert] /content/drive/MyDrive/ML_Fall_Detection/results/03_fehlalarme_2026-08-26_1102/results.json


PosixPath('/content/drive/MyDrive/ML_Fall_Detection/results/03_fehlalarme_2026-08-26_1102/results.json')